## Luís Forti - 3981932
## Marcell Slyuch - 3992616

# NLP - Text Classification Lab

Note that this lab has three levels: basic, regular and advanced.


Completing the **basic** part earns you a grade of 5.5-6.0.

Completing the **regular** part earns you a max grade of 8.0.

Completing the **advanced** part earns you a max grade of 10.0.

Please return a Jupyter notebook as a submission in Canvas, to make the grading easier for us.




## Basic Level

## Necessary Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import re
from nltk import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.stem.porter import *
from nltk.corpus import stopwords
from tqdm import tqdm

### Reading the Dataset

In [2]:
dataset=pd.read_csv("reviews.csv")
#excluding unnecessary columns
dataset=dataset[['sentiment', 'text']]
dataset = dataset.rename(columns={'sentiment': 'class'})
dataset

,class,text
0,neg,"Now, I won't deny that when I purchased this o..."
1,neg,"The saddest thing about this ""tribute"" is that..."
2,neg,Last night I decided to watch the prequel or s...
3,neg,I have to admit that i liked the first half of...
4,neg,I was not impressed about this film especially...
...,...,...
24995,pos,"This film is fun, if your a person who likes a..."
24996,pos,After seeing this film I feel like I know just...
24997,neg,first this deserves about 5 stars due to actin...
24998,neg,If you like films that ramble with little plot...


### Exercise 1
Split the dataset into two data structures (pandas frames), one for the reviews (our documents) (X) and one for the class (y).
Check that the lengths of both dataframes are equal

In [3]:
X = dataset[['text']]
y = dataset[['class']]

print('Size of x is : ' + str(X.shape[0]) + ' , size of y is ' + str(y.shape[0]))

Size of x is : 25000 , size of y is 25000


### Preprocessing
We now begin our preprocessing task, by lowercasing all of the documents, removing special characters and numerical values. Then we tokenize and stem our documents. 
### Exercise 2
Write a function to_lower(X) that takes a dataframe and returns its content in lower case.

In [4]:
def to_lower(X):
    X = X.copy()
    
    for col in X.select_dtypes(include="object").columns:
        X[col] = X[col].str.lower()
        
    return X

In [5]:
X=to_lower(X)
X.head()

,text
0,"now, i won't deny that when i purchased this o..."
1,"the saddest thing about this ""tribute"" is that..."
2,last night i decided to watch the prequel or s...
3,i have to admit that i liked the first half of...
4,i was not impressed about this film especially...


### Exercise 3 
Write a function clean_text(X), that takes the dataset as an input and removes all special characters and numerical values from it.

In [6]:
def clean_text(X):
    X = X.copy()

    #If it's an object -> if it's a string -> change anything that isn't a letter or whitespace for a whitespace
    for col in X.select_dtypes(include='object').columns:
        X[col] = X[col].apply(
            lambda text: re.sub(r'[^a-zA-Z\s]', '', text)
            if isinstance(text, str) else text
        )
    
    return X

In [7]:
X=clean_text(X)
X

,text
0,now i wont deny that when i purchased this off...
1,the saddest thing about this tribute is that a...
2,last night i decided to watch the prequel or s...
3,i have to admit that i liked the first half of...
4,i was not impressed about this film especially...
...,...
24995,this film is fun if your a person who likes a ...
24996,after seeing this film i feel like i know just...
24997,first this deserves about stars due to acting...
24998,if you like films that ramble with little plot...


### Exercise 4
Write a function tokenize(X) that takes a dataframe and returns the tokens in each row (document).

In [8]:
def tokenize(X):
    X = X.copy()

    #If it's an object -> if it's a string -> tokenize
    for col in X.select_dtypes(include='object').columns:
        X[col] = X[col].apply(
            lambda text: word_tokenize(text)
            if isinstance(text, str) else text
        )
    
    return X

In [9]:
X=tokenize(X)
X

,text
0,"[now, i, wont, deny, that, when, i, purchased,..."
1,"[the, saddest, thing, about, this, tribute, is..."
2,"[last, night, i, decided, to, watch, the, preq..."
3,"[i, have, to, admit, that, i, liked, the, firs..."
4,"[i, was, not, impressed, about, this, film, es..."
...,...
24995,"[this, film, is, fun, if, your, a, person, who..."
24996,"[after, seeing, this, film, i, feel, like, i, ..."
24997,"[first, this, deserves, about, stars, due, to,..."
24998,"[if, you, like, films, that, ramble, with, lit..."


### Exercise 5
Write a function remove_stop_words(X, stop_words), that takes a dataset X and the set of stop words you want removed from it. Your function should return the dataset, free of any common words.

In [10]:
def remove_stop_words(X, stop_words):
    X = X.copy()

    #If it's an object -> if it's a list -> remove every token of the list that's on the stop_words list
    for col in X.select_dtypes(include='object').columns:
        X[col] = X[col].apply(
            lambda tokens: [word for word in tokens if word not in stop_words]
            if isinstance(tokens, list) else tokens
        )
    
    return X

In [11]:
X = remove_stop_words(X,  stopwords.words('english'))
X

,text
0,"[wont, deny, purchased, ebay, high, expectatio..."
1,"[saddest, thing, tribute, almost, singers, inc..."
2,"[last, night, decided, watch, prequel, shall, ..."
3,"[admit, liked, first, half, sleepers, looked, ..."
4,"[impressed, film, especially, fact, went, cine..."
...,...
24995,"[film, fun, person, likes, good, campy, featur..."
24996,"[seeing, film, feel, like, know, little, bit, ..."
24997,"[first, deserves, stars, due, acting, would, g..."
24998,"[like, films, ramble, little, plot, exposition..."


### Exercise 6
Write a function stem(X), that takes a dataframe X and returns the stems of all the words in it.

You're free to choose any stemmer you want.

It's also possible to use a lemmatizer (lemmatization will be a lot slower!).

In [12]:
def stem(X):
    X = X.copy()
    stemmer = PorterStemmer()

    #For progress_apply to show the progress bar
    tqdm.pandas()

    #If it's an object -> if it's a list -> stem every token of the list
    for col in X.select_dtypes(include='object').columns:
        X[col] = X[col].progress_apply(
            lambda tokens: [stemmer.stem(word) for word in tokens]
            if isinstance(tokens, list) else tokens
        )
    
    return X

In [13]:
X=stem(X)
X

100%|██████████| 25000/25000 [00:34<00:00, 725.67it/s]


,text
0,"[wont, deni, purchas, ebay, high, expect, incr..."
1,"[saddest, thing, tribut, almost, singer, inclu..."
2,"[last, night, decid, watch, prequel, shall, sa..."
3,"[admit, like, first, half, sleeper, look, good..."
4,"[impress, film, especi, fact, went, cinema, fa..."
...,...
24995,"[film, fun, person, like, good, campi, featur,..."
24996,"[see, film, feel, like, know, littl, bit, usa,..."
24997,"[first, deserv, star, due, act, would, give, b..."
24998,"[like, film, rambl, littl, plot, exposit, spic..."


### Exercise 7
Having called a tokenizer and a stemmer on our dataset, the resulting rows are now of type list.
We need to convert them back to str, as our CountVectorizer expects a dataset where every document is a string. 
Define a function to_String(X), that takes your dataset and stitches back its rows back to the str format.

In [14]:
def to_String(X):
    X = X.copy()

    #If it's an object -> if it's a list -> join every token of the list to form a string
    for col in X.select_dtypes(include='object').columns:
        X[col] = X[col].apply(
            lambda tokens: ' '.join(tokens)
            if isinstance(tokens, list) else tokens
        )
    
    return X

In [15]:
X=to_String(X)
X

,text
0,wont deni purchas ebay high expect incred outo...
1,saddest thing tribut almost singer includ othe...
2,last night decid watch prequel shall say call ...
3,admit like first half sleeper look good act ev...
4,impress film especi fact went cinema famili go...
...,...
24995,film fun person like good campi featur film ev...
24996,see film feel like know littl bit usa david ly...
24997,first deserv star due act would give better su...
24998,like film rambl littl plot exposit spice kinki...


### Vector Space Model
Now that we preprocessed our corpus, we can proceed to vectorize it.
### Exercise 8
We can now use the [CountVectorizer](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html) from sklearn to create our document-term-matrix.

a. Create a document-term matrix from your dataset X, use min_df and max_df parameters to exclude words that appear in less than 10 documents, and words that appear in more than 99.5% of the documents. We want to keep only words of medium frequency, as stated in the lecture.

In [16]:
vectorizer = CountVectorizer(
    min_df=10,
    max_df=0.995
)

#X.iloc -> dataframe to array
X = vectorizer.fit_transform(X.iloc[:, 0])

b. Print the size and the contents of your vocab (feature space)

In [17]:
vocab = vectorizer.get_feature_names_out()

print(f"The size of the vocabulary is: {len(vocab)}")
print("The first 20 items in the vocab are:")
print(vocab[:20])

The size of the vocabulary is: 13823
The first 20 items in the vocab are:
['aag' 'aam' 'aaron' 'ab' 'abandon' 'abba' 'abbey' 'abbi' 'abbot' 'abbott'
 'abc' 'abduct' 'abe' 'abet' 'abhay' 'abhorr' 'abid' 'abigail' 'abil'
 'abject']


### Training our Logistic Regressor

### Exercise 9:
Before we dive into training our model, let's get our vector of true labels **y** into the right format.
Notice that by printing the contents of **y** below, what we get are the labels **neg** and **pos**. 
The model works only with **1 and 0**.
Let's convert the labels accordingly.

In [18]:
y[:10]

,class
0,neg
1,neg
2,neg
3,neg
4,neg
5,pos
6,pos
7,neg
8,pos
9,neg


a. Use the [label_ encoder](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.LabelEncoder.html) from Sklearn, to transform the labels in the vector **y** accordingly.

In [19]:
label_encoder = LabelEncoder()

#y.iloc -> dataframe to array
y = label_encoder.fit_transform(y.iloc[:, 0])

In [20]:
y[:10]

array([0, 0, 0, 0, 0, 1, 1, 0, 1, 0])

We also create a test and train set from our DTM and our vector y.

In [21]:
train_X, test_X, train_y, test_y = train_test_split(X, y, test_size=0.25)

### Exercise 10
Use the [Logistic Regression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) classifier from Sklearn to train your model, and check its accuracy on the test set

In [22]:
#Instantiate the model
model = LogisticRegression(max_iter=1000)

#Train the model
model.fit(train_X, train_y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [23]:
train_pred_y = model.predict(train_X)
test_pred_y = model.predict(test_X)

train_accuracy = accuracy_score(train_y, train_pred_y)
test_accuracy = accuracy_score(test_y, test_pred_y)

print(f"Train accuracy: {train_accuracy*100}")
print(f"Test accuracy: {test_accuracy*100}")

Train accuracy: 99.488
Test accuracy: 86.688


**Our results show a clear sign of overfitting**

## Regular 

Our main goal in the regular exercises is to provide a basic implementation for the logistic regression model.


We'll first define a function **initialize(X)** to get the initial vector of weights W and bias b.

We will then do our **forward_pass(X, W, b)** to get a vector of predictions.

We then write a function **gradient_descent(X, W, b, y, lr)** to get the updated vector of weights W and bias b.


We conclude this part by writing the model function, which calls all of the functions we defined, and proceed with the learning.

### Exercise 11
Write a function **initialize(X)**, which takes a DTM as an input and returns a vector of weights W and a scalar b for the bias. Both are initialized with some random values.

In [24]:
#This line was turning train_X from a sparse matrix to a dense matrix
#This didn't change the results, but the process in exercise 15 went from 1.3s/it to 70it/s
#train_X=train_X.toarray()
train_y=np.array(train_y)
train_y=train_y.reshape(train_y.shape[0],)

In [25]:
def initialize(X):
    #np.random.randn -> gaussian distribution of random numbers
    #This allows for both positive and negative values
    W = np.random.randn(X.shape[1])
    b = np.random.randn()
    
    return W, b

In [26]:
W,b = initialize(train_X)
print("Shape of the vector W is:", W.shape)

Shape of the vector W is: (13823,)


### Exercise 12
Write a function **forward_pass(X, W, b)** which takes the DTM X, the vector W and the bias b as inputs and returns a vector of predictions P.

Your function should implement the following equations

$$Z=X.W + b$$
$$P=\sigma{(Z)}=\frac{1}{1+e^{-Z}}$$

You can also implement the sigmoid as a separate function, 

In [27]:
def forward_pass(X, W, b):
    Z = X @ W + b
    P = 1 / (1 + np.exp(-Z))
    return P

In [28]:
P = forward_pass(train_X, W, b)
print("The vector of predictions shape is:", P.shape)

The vector of predictions shape is: (18750,)


### Exercise 13
Calculating the loss/cost function
You can use the [implementation](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html) by Sklearn, but feel free to also implement your own.

In [29]:
def cost_function(y, P):
    return log_loss(y, P)

cost = cost_function(train_y, P)
print(cost)

5.360835742823265


### Exercise 14
Write a function **gradient_descent(X, W, b , P, y, lr)** and returns the updated weight vector W, and bias b.
Your function needs to implement the following equations:

$$dW=\frac{1}{ne}X^T . (P-y)$$

$$db=\frac{1}{ne} \sum(P-y)$$

$$W=W-\alpha dW$$

$$b=b-\alpha db$$

In [30]:
def gradient_descent(X, W, b, P, y, lr):
    ne = X.shape[0]
    
    dW = (1 / ne) * X.T @ (P - y)
    db = (1 / ne) * np.sum(P - y)
    
    W = W - lr * dW
    b = b - lr * db
    return W, b

In [31]:
W, b = gradient_descent(train_X, W, b, P, train_y, lr=0.2)
print("shape of the updated W is:", W.shape)

shape of the updated W is: (13823,)


### Exercise 15
Now we can implement our logistic regression model in 3 simple steps
1. initialize the vector W and the bias b

2. repeat until number of iterations is reached   
    2.1. get a vector of predictions P  
    2.2. update the weights W and bias b using gradient descent  
    
3. return the final vector W and bias b

**logistic_regression(X, y, lr, iters)**, takes the matrix X as an input, the vector of true labels y, a learning rate, and the number of iterations iters. The function returns the learned parameters of the model, namely W and b.

In [32]:
def logistic_regression(X, y, lr, iters):
    W, b = initialize(X)

    for i in tqdm(range(iters)):
        P = forward_pass(X, W, b)
        W, b = gradient_descent(X, W, b, P, y, lr)
    
    return W, b

In [43]:
W, b = logistic_regression(train_X, train_y, lr=1.5, iters=500)

100%|██████████| 500/500 [00:06<00:00, 71.90it/s]


### Testing our model
We have been able to implement the model and run it on our training set. It's time to see how well it does. 
We'll first make a function **predict(X, W, b)**, that takes the dataset and the learned parameters and returns an array of predictions. Our threshold is 0.5, any prediction below that is returned as 0, and any above it are returned as 1.

In [44]:
def predict(X, W, b):
    P = forward_pass(X, W, b)
    P=1*(P >= 0.5)
    return P

In [45]:
P_train=predict(train_X, W, b)

In [46]:
print("Accuracy on our train set is, ",accuracy_score( train_y, P_train)*100, "%")

Accuracy on our train set is,  85.72266666666667 %


In [47]:
P_test=predict(test_X, W, b)

In [48]:
print("Accuracy on our test set is", accuracy_score(test_y, P_test)*100, "%")

Accuracy on our test set is 80.784 %


## Advanced

In this part, we set to understand what did the model actually learn.

### Exercise 16
Using the CountVectorizer of Sklearn, recreate a pandas frame where the rows contain the documents and the columns contain the features. 

In [49]:
DTM = pd.DataFrame(
    X.toarray(),
    columns=vectorizer.get_feature_names_out()
)

In [50]:
DTM

,aag,aam,aaron,ab,abandon,abba,abbey,abbi,abbot,abbott,...,zombi,zombiesbr,zone,zoo,zoom,zorro,zu,zucker,zulu,zuniga
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24996,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24997,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
24998,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Exercise 17
Knowing that our logistic regressor learns a weight for each feature (word in the vocab),  return the words with the highest weights (5 highest), and the words with lowest weights (5 lowest).

In [51]:
vocab = vectorizer.get_feature_names_out()

lowest_indices = np.argsort(W)[:5]
highest_indices = np.argsort(W)[-5:][::-1]

print("5 words with the lowest weights:")
for i in lowest_indices:
    print(vocab[i], W[i])

print("\n\n__________________________________\n\n")

print("5 words with the highest weights:")
for i in highest_indices:
    print(vocab[i], W[i])

5 words with the lowest weights:
worst -4.326006872601347
monoton -3.8128052275211846
craftsmanship -3.6416103802824744
boulder -3.5489454837247045
cooler -3.313570193912572


__________________________________


5 words with the highest weights:
saniti 3.523776290955342
screenplaybr 3.494575124424885
mulholland 3.4870795367432605
paget 3.397921120325971
heartfelt 3.348363096507844


### Exercise 18
Print the weights of the words "good" and "bad"

In [52]:
print("Weight of 'bad':", W[np.where(vocab == "bad")[0][0]])
print("Weight of 'good':", W[np.where(vocab == "good")[0][0]])

Weight of 'bad': -1.6985634134074112
Weight of 'good': 0.38328138496904995


##### Hope you enjoyed learning about logistic regression!